# D · Initialization and input-reference alignment controls

This experiment asks whether pretraining transfers useful information. An initialized encoder tests what the architecture and output training can achieve without encoder optimization. A shuffled-reference JEPA encoder tests whether correctly aligned reference features matter.

Read after notebooks 00–06. This notebook uses the prepared bundle and saved evaluation from that same run; the internal recipe group is `I`.


In [ ]:
from pathlib import Path
import json, os, sys

# Find the checkout/release from the notebook's working directory.
ROOT = Path(os.environ.get('GF_ROOT', Path.cwd())).resolve()
while not (ROOT / 'src/gavd6_sjepa').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/gavd6_sjepa').is_dir(), 'Open this notebook from the GAVD6 checkout or release.'
sys.path.insert(0, str(ROOT / 'notebooks/gait_fidelity'))
sys.path.insert(0, str(ROOT / 'src'))
from tutorial_helpers import configure, preview_images
study = configure(ROOT)


## Inspect the exact recipe cells

Compare each control with the prespecified graph-time paired-JEPA recipe under the same base or paired-change output objective. Initialized encoders have no pretraining mask or loss. All output networks receive the same unmasked deployment-style observed inputs and independently initialized output parameters.


In [ ]:
import pandas as pd
plan = study.artifact('plan.json')
group_recipes = [r for r in plan['recipes'] if r['group'] == 'I']
expected_count = 4 if plan.get('experiment_set', 'full') == 'full' else {'M': 4, 'T': 0, 'P': 2, 'I': 4, 'L': 0}['I']
assert len(group_recipes) == expected_count, 'Saved plan differs from the selected experiment set.'
recipe_ids = {r['recipe_id'] for r in group_recipes}
display(pd.DataFrame(group_recipes))
print('Final models:', len(group_recipes) * len(plan['seeds']), 'Seeds:', plan['seeds'])
if not group_recipes:
    print('This group is outside the saved core protocol. Its worked examples are educational; no results are implied.')


## Construct a shuffled-reference control without crossing people or splits

A donor replaces only the reference window used in feature pretraining.
Donors keep the training person, physical state, camera, naming condition,
observation, extractor, movement role and magnitude. Their source family
must differ. A bijection within each such stratum preserves reference
frequency. This retains nuisance information while removing the correct
input-to-reference temporal correspondence.

Below we reproduce the production donor assignment as short cycles across
source families. Pairing two distinct families, with one three-cycle when
needed, avoids a quadratic assignment matrix as the cohort grows. Seeded
tie-breaking makes the choice reproducible. Development
rows do not enter this calculation. Initialization-only encoders have no
donor selection because they have no pretraining phase.


In [ ]:
import numpy as np
from collections import defaultdict
import heapq
from gavd6_sjepa.research_directions.gait_fidelity.data import load_dataset
from gavd6_sjepa.research_directions.gait_fidelity.training import shuffled_reference_indices
bundle = load_dataset(study.bundle_path())
records = bundle.records
fields = ['canonical_person_id', 'physical_state', 'camera_id', 'naming',
          'observation', 'extractor', 'movement_state', 'movement_magnitude']
strata = defaultdict(list)
for index, record in enumerate(records):
    if record['split'] == 'train':
        strata[tuple(str(record.get(key)) for key in fields)].append(index)
seed = plan['seeds'][0]
rng = np.random.default_rng(seed + 65537)
donors = np.arange(len(records))
for indices in strata.values():
    families = np.array([records[i]['source_family_id'] for i in indices])
    groups = defaultdict(list)
    for local_index, family_id in enumerate(families):
        groups[str(family_id)].append(local_index)
    count = len(indices)
    assert count >= 2 and max(map(len, groups.values())) <= count // 2
    for members in groups.values(): rng.shuffle(members)
    permutation = np.full(count, -1, dtype=np.int64)
    heap = [(-len(members), float(rng.random()), key) for key, members in groups.items()]
    heapq.heapify(heap)
    def largest(number):
        return [heapq.heappop(heap)[2] for _ in range(min(number, len(heap)))]
    def replace(keys):
        for key in keys:
            if groups[key]: heapq.heappush(heap, (-len(groups[key]), float(rng.random()), key))
    if count % 2:
        keys = largest(3)
        assert len(keys) == 3
        cycle = [groups[key].pop() for key in keys]
        if rng.random() < .5: cycle.reverse()
        permutation[cycle] = np.roll(cycle, -1)
        replace(keys)
    while heap:
        keys = largest(2)
        assert len(keys) == 2
        a, b = [groups[key].pop() for key in keys]
        permutation[a], permutation[b] = b, a
        replace(keys)
    assert np.all(families != families[permutation])
    assert len(np.unique(permutation)) == count
    donors[indices] = np.asarray(indices)[permutation]
np.testing.assert_array_equal(donors, shuffled_reference_indices(records, seed))
example_rows = next(iter(strata.values()))
display(pd.DataFrame([{'input_person': records[i]['canonical_person_id'],
                      'input_family': records[i]['source_family_id'],
                      'reference_family': records[donors[i]]['source_family_id'],
                      'split': records[donors[i]]['split']} for i in example_rows]))
assert all(donors[i] == i for i, r in enumerate(records) if r['split'] != 'train')


Notebook 04 shows which weights change during feature training and which
remain fixed during output training. Compare these controls with graph-time
paired JEPA under the same output loss and seed. Similar downstream errors
would limit evidence that aligned feature prediction contributes useful
information; they would not establish that the learned representations are
identical. Any favorable difference still needs the direct-training and
coordinate-pretraining benchmarks.


## Follow the shared dependencies

This tutorial inspects the existing central queue. It does not launch a separate copy of its group: that would duplicate pretraining and break the global budget. Notebook 04 launches all groups, and this table identifies the phases that belong to the present comparison.


In [ ]:
final_phases = [p for p in plan['phases'] if p['phase'] != 'pretrain' and p['recipe']['recipe_id'] in recipe_ids]
parent_ids = {parent for p in final_phases for parent in p['depends_on'] if parent != 'prepare'}
selected = [p for p in plan['phases'] if p in final_phases or p['phase_id'] in parent_ids]
display(pd.DataFrame([{'phase_id': p['phase_id'], 'phase': p['phase'], 'seed': p['seed'],
                      'depends_on': ', '.join(p['depends_on'])} for p in selected]))


## Inspect completed checkpoints and learning histories

Each completed phase links its retained checkpoint, history and predictions to its source identity. Missing phases are reported as pending; a checkpoint from another study is not substituted.


In [ ]:
ledger_path = study.work / 'ledger.json'
completed = json.loads(ledger_path.read_text()).get('completed', {}) if ledger_path.exists() else {}
rows = []
for phase in selected:
    saved = completed.get(phase['phase_id'])
    result = saved.get('result', {}) if saved else {}
    rows.append({'phase_id': phase['phase_id'], 'status': 'complete' if saved else 'pending',
                 'checkpoint': result.get('checkpoint'), 'predictions': result.get('predictions')})
display(pd.DataFrame(rows))
history_candidates = []
for row in rows:
    if row['checkpoint']:
        history_path = Path(row['checkpoint']).parent / 'history.json'
        if history_path.exists(): history_candidates.append(history_path)
if history_candidates:
    history_path = history_candidates[0]
    print('First declared completed history:', history_path)
    display(pd.DataFrame(json.loads(history_path.read_text())))
else:
    print('No completed histories yet. Run or resume the central queue from notebook 04.')


## Read group results on the common population

Check that the shuffled-reference source window differs from the input family and does not cross split boundaries. These controls attribute findings to the graph-time comparison only; they do not establish that every mask benefits equally from alignment or pretraining.


In [ ]:
person_path = study.work / 'evaluation/per-person.csv'
if person_path.exists():
    people = pd.read_csv(person_path)
    display(people.loc[people['method'].isin(recipe_ids)])
    coverage_path = study.work / 'evaluation/coverage.csv'
    if coverage_path.exists():
        coverage = pd.read_csv(coverage_path)
        display(coverage.loc[coverage['method'].isin(recipe_ids)])
else:
    print('Evaluation is pending. These recipe cards do not fabricate or extrapolate results.')


Read the matching controls from the other experiment tutorials before attribution. Full evaluation and numerical reconstruction are covered by notebooks 05 and 06.
